# Preprocessing Climate Data

`.nc` is a netCDF4 file

In [90]:
import netCDF4
import numpy as np
import torch

## Downlaod the data from NOAA

Note: the precise filename will change as the data is updated by NOAA. Just go to https://www.ncei.noaa.gov/data/mean-layer-temperature-noaa/access/ and grab the latest TLT (lower troposphere) link.

In [1]:
!wget -nc https://www.ncei.noaa.gov/data/mean-layer-temperature-noaa/access/Mean-Layer-Temperature-NOAA_v05r00_TLT_S198101_E202412_C20250105.nc

--2025-02-12 09:20:15--  https://www.ncei.noaa.gov/data/mean-layer-temperature-noaa/access/Mean-Layer-Temperature-NOAA_v05r00_TLT_S198101_E202501_C20250206.nc
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.167, 205.167.25.177, 205.167.25.171, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 33781686 (32M) [application/x-netcdf]
Saving to: ‘Mean-Layer-Temperature-NOAA_v05r00_TLT_S198101_E202501_C20250206.nc’

Mean-Layer-Temperat 100%[===================>]  32.22M  3.38MB/s    in 9.4s    

2025-02-12 09:20:25 (3.43 MB/s) - ‘Mean-Layer-Temperature-NOAA_v05r00_TLT_S198101_E202501_C20250206.nc’ saved [33781686/33781686]



In [91]:
temp_dataset = netCDF4.Dataset("Mean-Layer-Temperature-NOAA_v05r00_TLT_S198101_E202412_C20250105.nc")

In [92]:
# Uncomment this line to get a dump of all the variables in the dataset
# temp_dataset.variables

In [93]:
all_temps_kelvin = temp_dataset.variables["tcdr_MSU_AMSUA_ATMS_TLT"][:]
all_temps_kelvin.shape

In [94]:
all_lons, all_lats = np.meshgrid(np.linspace(-180, 180, all_temps_kelvin.shape[2]+1), 
                         np.linspace(-90, 90, all_temps_kelvin.shape[1]+1))

In [95]:
# Drop latitudes with no data (|latitude| > 82.5, first and last three bins)
temps_kelvin = all_temps_kelvin[:, 3:-3, :]
lons = all_lons[3:-3, :]
lats = all_lats[3:-3, :]
num_lats, num_lons = lats.shape
num_locs = lons.size
print(f"latitude range: {lats.min()} - {lats.max()}")

latitude range: -82.5 - 82.5


In [96]:
dates = temp_dataset.variables["time"][:]
start = np.datetime64('1978-01-01')
dates = start + np.array([np.timedelta64(int(d), 'D') for d in dates])
num_timesteps = len(dates)
print(dates[0], dates[-1])


## Save pytorch tensors

In [101]:
torch.save(dict(temperature=torch.tensor(temps_kelvin.data, dtype=torch.float32),
                dates=dates,
                longitude=torch.tensor(lons, dtype=torch.float32),
                latitude=torch.tensor(lats, dtype=torch.float32)), 
                'hw3.pt')

In [102]:
# Make sure we can load the data back in
hw3 = torch.load('hw3.pt', weights_only=False)
print(hw3.keys())

dict_keys(['temperature', 'dates', 'longitude', 'latitude'])


tensor([[-82.5000, -82.5000, -82.5000,  ..., -82.5000, -82.5000, -82.5000],
        [-80.0000, -80.0000, -80.0000,  ..., -80.0000, -80.0000, -80.0000],
        [-77.5000, -77.5000, -77.5000,  ..., -77.5000, -77.5000, -77.5000],
        ...,
        [ 77.5000,  77.5000,  77.5000,  ...,  77.5000,  77.5000,  77.5000],
        [ 80.0000,  80.0000,  80.0000,  ...,  80.0000,  80.0000,  80.0000],
        [ 82.5000,  82.5000,  82.5000,  ...,  82.5000,  82.5000,  82.5000]])